In [5]:
import os
import re
import io
import requests
import zipfile
import pandas as pd
import numpy as np

In [6]:
company_df = pd.read_csv("../data/sp500_companies.csv")

company_df

,ticker,company_name,sector,subsector,cik
0,MMM,3M,Industrials,Industrial Conglomerates,66740
1,AOS,A. O. Smith,Industrials,Building Products,91142
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,1800
3,ABBV,AbbVie,Health Care,Biotechnology,1551152
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,1467373
...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,1524472
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,1041061
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,877212
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,1136869


In [41]:
def download_gdelt_gkg(timestamp):
    """
    timestamp format: YYYYMMDDHHMMSS (e.g. '20220103000000')
    """
    url = f"http://data.gdeltproject.org/gdeltv2/{timestamp}.gkg.csv.zip"
    try:
        r = requests.get(url, timeout=20)
        if r.status_code != 200:
            return None
        
        z = zipfile.ZipFile(io.BytesIO(r.content))
        df = pd.read_csv(
            z.open(z.namelist()[0]),
            sep="\t",
            header=None,
            low_memory=False,
            dtype=str,
            encoding="latin-1"   # IMPORTANT FIX
        )
        return df
    
    except Exception as e:
        print("Download error:", e)
        return None

def clean_url_to_text(url):
    if not isinstance(url, str):
        return ""
    url = url.replace("https://", "").replace("http://", "")
    url = re.sub(r'^[^/]+/', '', url)         # remove domain
    url = re.sub(r'[-_/]', ' ', url)         # replace -, _, / with spaces
    url = re.sub(r'[^a-zA-Z0-9 ]+', ' ', url)
    url = re.sub(r'\s+', ' ', url)
    return url.strip().lower()


def clean_snippet(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-zA-Z ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

def filter_company_news(gkg_df, company_df, date_str):
    results = []
    if gkg_df is None:
        return results

    # GDELT column references based on your file:
    name_list = gkg_df[1].astype(str).str.lower()
    url_main = gkg_df[4].astype(str).str.lower()
    url_alt  = gkg_df[26].astype(str).str.lower()
    snippet  = gkg_df[24].astype(str).str.lower()

    for _, c in company_df.iterrows():
        ticker = c['ticker']
        company = str(c['company_name']).lower()
        key = company.split()[0]     # first word of company name

        mask = (
            name_list.str.contains(key, na=False) |
            url_main.str.contains(key, na=False) |
            url_alt.str.contains(key, na=False) |
            snippet.str.contains(key, na=False)
        )

        matched = gkg_df[mask]

        for _, row in matched.iterrows():
            results.append({
                'date': date_str,
                'ticker': ticker,
                'url': row[4],           # MAIN URL
                'alt_url': row[26],      # alt URL
                'snippet': row[24],      # snippet field
                'headline': row[4]       # treat URL as headline base
            })

    return results


In [42]:
test_tickers = ["AAPL", "MSFT", "AMZN"]
test_company_df = company_df[company_df['ticker'].isin(test_tickers)]
test_company_df


,ticker,company_name,sector,subsector,cik
22,AMZN,Amazon,Consumer Discretionary,Broadline Retail,1018724
38,AAPL,Apple Inc.,Information Technology,"Technology Hardware, Storage & Peripherals",320193
316,MSFT,Microsoft,Information Technology,Systems Software,789019


In [43]:
test_date = "2022-01-03"
timestamp = test_date.replace("-", "") + "000000"

gkg = download_gdelt_gkg(timestamp)
rows = filter_company_news(gkg, test_company_df, test_date.replace("-", ""))
rows[:5], len(rows)


([{'date': '20220103',
   'ticker': 'AMZN',
   'url': 'https://www.cityandstateny.com/politics/2022/01/police-officer-mayors-first-day-job/360258/',
   'alt_url': "<PAGE_LINKS>https://compstat.nypdonline.org/2e5c3f4b-85c1-4635-83c6-22b27fe7c75c/view/89;https://en.wikipedia.org/wiki/Kosciuszko_Street_station;https://gothamist.com/news/eric-adams-sworn-110th-new-york-city-mayor;https://nypost.com/2021/12/30/nyc-social-services-provider-blew-city-money-on-staff-booze-cruise-fast-food-audit/;https://nypost.com/2022/01/01/eric-adams-makes-911-call-on-first-day-as/;https://nypost.com/2022/01/01/nyc-recorded-485-murders-in-2021/;https://twitter.com/MylesMill/status/1477264288964296707;https://twitter.com/NYPDnews/status/1477345880080482304;https://twitter.com/courtneycgross/status/1477263721445699586;https://twitter.com/courtneycgross/status/1477264329162506240;https://twitter.com/courtneycgross/status/1477268541044838403;https://twitter.com/courtneycgross/status/1477268987889303552;https://t

In [11]:
news_raw = pd.DataFrame(rows)
news_raw.head()


,date,ticker,url,alt_url,snippet,headline
0,20220103,AMZN,https://www.cityandstateny.com/politics/2022/0...,<PAGE_LINKS>https://compstat.nypdonline.org/2e...,"2,large,2444;33,staff members joined on,4145;9...",https://www.cityandstateny.com/politics/2022/0...
1,20220103,AMZN,https://listverse.com/2015/06/13/10-successful...,<PAGE_LINKS>http://articles.latimes.com/1987-1...,"10,children,602;7,of whom survived into,616;3,...",https://listverse.com/2015/06/13/10-successful...
2,20220103,AAPL,https://www.nbcdfw.com/news/politics/twitter-p...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"3,strikes earn a 12-hour,493;2,committee assig...",https://www.nbcdfw.com/news/politics/twitter-p...
3,20220103,AAPL,https://www.nbcdfw.com/news/business/money-rep...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"80,points,129;500,added 0 3%,152;100,futures g...",https://www.nbcdfw.com/news/business/money-rep...
4,20220103,AAPL,https://appleinsider.com/articles/22/01/02/the...,<PAGE_LINKS>https://appleinsider.com/inside/io...,"35,health,3724;",https://appleinsider.com/articles/22/01/02/the...


In [12]:
news_raw['headline_text'] = news_raw['url'].apply(clean_url_to_text)
news_raw['snippet_text'] = news_raw['snippet'].apply(clean_snippet)

# Combined text for FinBERT
news_raw['combined_text'] = (
    news_raw['headline_text'] + " " + news_raw['snippet_text']
).str.strip()

news_raw.head()


,date,ticker,url,alt_url,snippet,headline,headline_text,snippet_text,combined_text
0,20220103,AMZN,https://www.cityandstateny.com/politics/2022/0...,<PAGE_LINKS>https://compstat.nypdonline.org/2e...,"2,large,2444;33,staff members joined on,4145;9...",https://www.cityandstateny.com/politics/2022/0...,politics 2022 01 police officer mayors first d...,large staff members joined on dreams murders w...,politics 2022 01 police officer mayors first d...
1,20220103,AMZN,https://listverse.com/2015/06/13/10-successful...,<PAGE_LINKS>http://articles.latimes.com/1987-1...,"10,children,602;7,of whom survived into,616;3,...",https://listverse.com/2015/06/13/10-successful...,2015 06 13 10 successful people trapped in the...,children of whom survived into of darwin child...,2015 06 13 10 successful people trapped in the...
2,20220103,AAPL,https://www.nbcdfw.com/news/politics/twitter-p...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"3,strikes earn a 12-hour,493;2,committee assig...",https://www.nbcdfw.com/news/politics/twitter-p...,news politics twitter permanently suspends mar...,strikes earn a hour committee assignments of a...,news politics twitter permanently suspends mar...
3,20220103,AAPL,https://www.nbcdfw.com/news/business/money-rep...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"80,points,129;500,added 0 3%,152;100,futures g...",https://www.nbcdfw.com/news/business/money-rep...,news business money report stock futures rise ...,points added futures gained rose nearly for de...,news business money report stock futures rise ...
4,20220103,AAPL,https://appleinsider.com/articles/22/01/02/the...,<PAGE_LINKS>https://appleinsider.com/inside/io...,"35,health,3724;",https://appleinsider.com/articles/22/01/02/the...,articles 22 01 02 the best iphone widget apps ...,health,articles 22 01 02 the best iphone widget apps ...


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device)
model.eval()


/home/zeyuzh/.conda/envs/jupyter_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on: cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [14]:
def finbert_gpu(texts, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        
        with torch.no_grad():
            logits = model(**enc).logits
        
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)

    return np.vstack(all_probs)


In [15]:
probs = finbert_gpu(news_raw['combined_text'].tolist())
news_raw['neg'] = probs[:,0]
news_raw['neu'] = probs[:,1]
news_raw['pos'] = probs[:,2]


In [16]:
news_raw.head()


,date,ticker,url,alt_url,snippet,headline,headline_text,snippet_text,combined_text,neg,neu,pos
0,20220103,AMZN,https://www.cityandstateny.com/politics/2022/0...,<PAGE_LINKS>https://compstat.nypdonline.org/2e...,"2,large,2444;33,staff members joined on,4145;9...",https://www.cityandstateny.com/politics/2022/0...,politics 2022 01 police officer mayors first d...,large staff members joined on dreams murders w...,politics 2022 01 police officer mayors first d...,0.146713,0.254010,0.599277
1,20220103,AMZN,https://listverse.com/2015/06/13/10-successful...,<PAGE_LINKS>http://articles.latimes.com/1987-1...,"10,children,602;7,of whom survived into,616;3,...",https://listverse.com/2015/06/13/10-successful...,2015 06 13 10 successful people trapped in the...,children of whom survived into of darwin child...,2015 06 13 10 successful people trapped in the...,0.043823,0.086343,0.869834
2,20220103,AAPL,https://www.nbcdfw.com/news/politics/twitter-p...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"3,strikes earn a 12-hour,493;2,committee assig...",https://www.nbcdfw.com/news/politics/twitter-p...,news politics twitter permanently suspends mar...,strikes earn a hour committee assignments of a...,news politics twitter permanently suspends mar...,0.011281,0.880396,0.108323
3,20220103,AAPL,https://www.nbcdfw.com/news/business/money-rep...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"80,points,129;500,added 0 3%,152;100,futures g...",https://www.nbcdfw.com/news/business/money-rep...,news business money report stock futures rise ...,points added futures gained rose nearly for de...,news business money report stock futures rise ...,0.787802,0.103075,0.109123
4,20220103,AAPL,https://appleinsider.com/articles/22/01/02/the...,<PAGE_LINKS>https://appleinsider.com/inside/io...,"35,health,3724;",https://appleinsider.com/articles/22/01/02/the...,articles 22 01 02 the best iphone widget apps ...,health,articles 22 01 02 the best iphone widget apps ...,0.113584,0.012578,0.873838


In [17]:
daily_news = (
    news_raw.groupby(['date','ticker'])
    .agg(
        sent_mean=('pos', 'mean'),
        sent_vol=('pos', 'std'),
        pos_ratio=('pos', lambda x: (x > 0.5).mean()),
        neg_ratio=('neg', lambda x: (x > 0.5).mean()),
        neu_ratio=('neu', lambda x: (x > 0.5).mean()),
        news_count=('combined_text', 'count')
    )
    .fillna(0)
    .reset_index()
)

daily_news


,date,ticker,sent_mean,sent_vol,pos_ratio,neg_ratio,neu_ratio,news_count
0,20220103,AAPL,0.627835,0.345108,0.75,0.125,0.125,8
1,20220103,AMZN,0.734556,0.191313,1.00,0.000,0.000,2
2,20220103,MSFT,0.524660,0.561535,0.50,0.000,0.500,2


In [18]:
date_str = "20220103"
daily_rows = []

for hour in range(24):
    ts = f"{date_str}{hour:02d}0000"
    gkg = download_gdelt_gkg(ts)
    if gkg is not None:
        rows = filter_company_news(gkg, test_company_df, date_str)
        daily_rows.extend(rows)

news_raw = pd.DataFrame(daily_rows)


In [19]:
news_raw

,date,ticker,url,alt_url,snippet,headline
0,20220103,AMZN,https://www.cityandstateny.com/politics/2022/0...,<PAGE_LINKS>https://compstat.nypdonline.org/2e...,"2,large,2444;33,staff members joined on,4145;9...",https://www.cityandstateny.com/politics/2022/0...
1,20220103,AMZN,https://listverse.com/2015/06/13/10-successful...,<PAGE_LINKS>http://articles.latimes.com/1987-1...,"10,children,602;7,of whom survived into,616;3,...",https://listverse.com/2015/06/13/10-successful...
2,20220103,AAPL,https://www.nbcdfw.com/news/politics/twitter-p...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"3,strikes earn a 12-hour,493;2,committee assig...",https://www.nbcdfw.com/news/politics/twitter-p...
3,20220103,AAPL,https://www.nbcdfw.com/news/business/money-rep...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"80,points,129;500,added 0 3%,152;100,futures g...",https://www.nbcdfw.com/news/business/money-rep...
4,20220103,AAPL,https://appleinsider.com/articles/22/01/02/the...,<PAGE_LINKS>https://appleinsider.com/inside/io...,"35,health,3724;",https://appleinsider.com/articles/22/01/02/the...
...,...,...,...,...,...,...
525,20220103,AAPL,https://www.kmaj.com/2021/06/15/this-months-cr...,<PAGE_LINKS>https://www.kmaj.com/2019/01/09/cr...,"1,Math teacher at Rossville,336;1,dollars ,977...",https://www.kmaj.com/2021/06/15/this-months-cr...
526,20220103,AAPL,https://www.nytimes.com/2022/01/03/briefing/om...,<PAGE_LINKS>http://www.nytimes.com/newsletters...,"3000000000000,dollars company,1170;10000000000...",https://www.nytimes.com/2022/01/03/briefing/om...
527,20220103,AAPL,https://www.gizchina.com/2022/01/03/iphone-14-...,<PAGE_LINKS>http://www.apple.com;https://www.g...,"14,series,59;14,series will come with,253;14,s...",https://www.gizchina.com/2022/01/03/iphone-14-...
528,20220103,AAPL,https://ricochet.com/podcast/sara-carter-podca...,<PAGE_LINKS>https://itunes.apple.com/us/podcas...,NaN,https://ricochet.com/podcast/sara-carter-podca...


In [22]:
all_rows = []

test_tickers = ["AAPL", "MSFT", "AMZN"]

test_start = "2022-01-03"
test_end   = "2022-01-10"
test_dates = pd.date_range(test_start, test_end, freq="D")
for d in test_dates:
    date_str = d.strftime("%Y%m%d")
    for hour in range(24):
        ts = f"{date_str}{hour:02d}0000"
        gkg = download_gdelt_gkg(ts)
        if gkg is not None:
            rows = filter_company_news(gkg, test_company_df, date_str)
            all_rows.extend(rows)

news_raw = pd.DataFrame(all_rows)


In [23]:
news_raw

,date,ticker,url,alt_url,snippet,headline
0,20220103,AMZN,https://www.cityandstateny.com/politics/2022/0...,<PAGE_LINKS>https://compstat.nypdonline.org/2e...,"2,large,2444;33,staff members joined on,4145;9...",https://www.cityandstateny.com/politics/2022/0...
1,20220103,AMZN,https://listverse.com/2015/06/13/10-successful...,<PAGE_LINKS>http://articles.latimes.com/1987-1...,"10,children,602;7,of whom survived into,616;3,...",https://listverse.com/2015/06/13/10-successful...
2,20220103,AAPL,https://www.nbcdfw.com/news/politics/twitter-p...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"3,strikes earn a 12-hour,493;2,committee assig...",https://www.nbcdfw.com/news/politics/twitter-p...
3,20220103,AAPL,https://www.nbcdfw.com/news/business/money-rep...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"80,points,129;500,added 0 3%,152;100,futures g...",https://www.nbcdfw.com/news/business/money-rep...
4,20220103,AAPL,https://appleinsider.com/articles/22/01/02/the...,<PAGE_LINKS>https://appleinsider.com/inside/io...,"35,health,3724;",https://appleinsider.com/articles/22/01/02/the...
...,...,...,...,...,...,...
4609,20220110,MSFT,https://www.windowscentral.com/forza-street-cl...,<PAGE_LINKS>https://support.forzamotorsport.ne...,"5,is now available,1906;5,is available through...",https://www.windowscentral.com/forza-street-cl...
4610,20220110,MSFT,http://www.consumerelectronicsnet.com/f5-adds-...,<PAGE_LINKS>https://twitter.com/F5&esheet=5256...,"11,members,1799;10,of whom,1809;3,directors wh...",http://www.consumerelectronicsnet.com/f5-adds-...
4611,20220110,MSFT,https://www.windowscentral.com/phil-spencer-xb...,<PAGE_LINKS>https://www.nytimes.com/2022/01/10...,NaN,https://www.windowscentral.com/phil-spencer-xb...
4612,20220110,MSFT,https://www.esecurityplanet.com/threats/creden...,<PAGE_LINKS>https://ag.ny.gov/press-release/20...,"1000000,online accounts at 17,361;17,companies...",https://www.esecurityplanet.com/threats/creden...


In [24]:
news_raw['headline_text'] = news_raw['url'].apply(clean_url_to_text)
news_raw['snippet_text']  = news_raw['snippet'].apply(clean_snippet)
news_raw['combined_text'] = news_raw['headline_text'] + " " + news_raw['snippet_text']
news_raw.head()


,date,ticker,url,alt_url,snippet,headline,headline_text,snippet_text,combined_text
0,20220103,AMZN,https://www.cityandstateny.com/politics/2022/0...,<PAGE_LINKS>https://compstat.nypdonline.org/2e...,"2,large,2444;33,staff members joined on,4145;9...",https://www.cityandstateny.com/politics/2022/0...,politics 2022 01 police officer mayors first d...,large staff members joined on dreams murders w...,politics 2022 01 police officer mayors first d...
1,20220103,AMZN,https://listverse.com/2015/06/13/10-successful...,<PAGE_LINKS>http://articles.latimes.com/1987-1...,"10,children,602;7,of whom survived into,616;3,...",https://listverse.com/2015/06/13/10-successful...,2015 06 13 10 successful people trapped in the...,children of whom survived into of darwin child...,2015 06 13 10 successful people trapped in the...
2,20220103,AAPL,https://www.nbcdfw.com/news/politics/twitter-p...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"3,strikes earn a 12-hour,493;2,committee assig...",https://www.nbcdfw.com/news/politics/twitter-p...,news politics twitter permanently suspends mar...,strikes earn a hour committee assignments of a...,news politics twitter permanently suspends mar...
3,20220103,AAPL,https://www.nbcdfw.com/news/business/money-rep...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"80,points,129;500,added 0 3%,152;100,futures g...",https://www.nbcdfw.com/news/business/money-rep...,news business money report stock futures rise ...,points added futures gained rose nearly for de...,news business money report stock futures rise ...
4,20220103,AAPL,https://appleinsider.com/articles/22/01/02/the...,<PAGE_LINKS>https://appleinsider.com/inside/io...,"35,health,3724;",https://appleinsider.com/articles/22/01/02/the...,articles 22 01 02 the best iphone widget apps ...,health,articles 22 01 02 the best iphone widget apps ...


In [25]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device)
model.eval()


Running on: cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [26]:
def finbert_gpu(texts, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            logits = model(**enc).logits
        
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
    
    return np.vstack(all_probs)


In [27]:
probs = finbert_gpu(news_raw['combined_text'].tolist())

news_raw['neg'] = probs[:,0]
news_raw['neu'] = probs[:,1]
news_raw['pos'] = probs[:,2]

news_raw.head()


,date,ticker,url,alt_url,snippet,headline,headline_text,snippet_text,combined_text,neg,neu,pos
0,20220103,AMZN,https://www.cityandstateny.com/politics/2022/0...,<PAGE_LINKS>https://compstat.nypdonline.org/2e...,"2,large,2444;33,staff members joined on,4145;9...",https://www.cityandstateny.com/politics/2022/0...,politics 2022 01 police officer mayors first d...,large staff members joined on dreams murders w...,politics 2022 01 police officer mayors first d...,0.146713,0.254009,0.599277
1,20220103,AMZN,https://listverse.com/2015/06/13/10-successful...,<PAGE_LINKS>http://articles.latimes.com/1987-1...,"10,children,602;7,of whom survived into,616;3,...",https://listverse.com/2015/06/13/10-successful...,2015 06 13 10 successful people trapped in the...,children of whom survived into of darwin child...,2015 06 13 10 successful people trapped in the...,0.043823,0.086343,0.869834
2,20220103,AAPL,https://www.nbcdfw.com/news/politics/twitter-p...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"3,strikes earn a 12-hour,493;2,committee assig...",https://www.nbcdfw.com/news/politics/twitter-p...,news politics twitter permanently suspends mar...,strikes earn a hour committee assignments of a...,news politics twitter permanently suspends mar...,0.011281,0.880396,0.108323
3,20220103,AAPL,https://www.nbcdfw.com/news/business/money-rep...,<PAGE_LINKS>https://apps.apple.com/app/id33180...,"80,points,129;500,added 0 3%,152;100,futures g...",https://www.nbcdfw.com/news/business/money-rep...,news business money report stock futures rise ...,points added futures gained rose nearly for de...,news business money report stock futures rise ...,0.787802,0.103075,0.109123
4,20220103,AAPL,https://appleinsider.com/articles/22/01/02/the...,<PAGE_LINKS>https://appleinsider.com/inside/io...,"35,health,3724;",https://appleinsider.com/articles/22/01/02/the...,articles 22 01 02 the best iphone widget apps ...,health,articles 22 01 02 the best iphone widget apps ...,0.113584,0.012578,0.873838


In [28]:
daily_news = (
    news_raw.groupby(['date','ticker'])
    .agg(
        sent_mean=('pos', 'mean'),
        sent_vol=('pos', 'std'),
        pos_ratio=('pos', lambda x: (x > 0.5).mean()),
        neg_ratio=('neg', lambda x: (x > 0.5).mean()),
        neu_ratio=('neu', lambda x: (x > 0.5).mean()),
        news_count=('combined_text', 'count')
    )
    .fillna(0)
    .reset_index()
)

daily_news


,date,ticker,sent_mean,sent_vol,pos_ratio,neg_ratio,neu_ratio,news_count
0,20220103,AAPL,0.650454,0.330671,0.726115,0.098726,0.162420,314
1,20220103,AMZN,0.807229,0.202903,0.901734,0.023121,0.069364,173
2,20220103,MSFT,0.757459,0.246255,0.860465,0.046512,0.093023,43
3,20220104,AAPL,0.595235,0.321596,0.621469,0.135593,0.216573,531
4,20220104,AMZN,0.769795,0.252340,0.863454,0.008032,0.120482,249
5,20220104,MSFT,0.659336,0.342644,0.704545,0.227273,0.045455,44
6,20220105,AAPL,0.659324,0.296977,0.721839,0.034483,0.216092,435
7,20220105,AMZN,0.772545,0.255051,0.861702,0.024823,0.106383,282
8,20220105,MSFT,0.766895,0.222253,0.868852,0.049180,0.081967,61
9,20220106,AAPL,0.673179,0.312200,0.724796,0.043597,0.201635,367
